In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img = cv2.imread("scenery.jpg", 0)
h, w = img.shape
lbp = np.zeros((h - 2, w - 2), dtype=np.uint8)

for i in range(1, h - 1):
    for j in range(1, w - 1):
        center = img[i, j]
        code = 0
        code |= (img[i-1, j-1] >= center) << 7
        code |= (img[i-1, j] >= center) << 6
        code |= (img[i-1, j+1] >= center) << 5
        code |= (img[i, j+1] >= center) << 4
        code |= (img[i+1, j+1] >= center) << 3
        code |= (img[i+1, j] >= center) << 2
        code |= (img[i+1, j-1] >= center) << 1
        code |= (img[i, j-1] >= center) << 0
        lbp[i-1, j-1] = code

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(img, cmap='gray')
plt.title("Original Grayscale")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(lbp, cmap='gray')
plt.title("Local Binary Pattern")
plt.axis("off")

plt.show()


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import wiener

img = cv2.imread("scenery.jpg", 0)

gauss_noise = np.random.normal(0, 25, img.shape)
noisy_gauss = np.clip(img.astype(np.float32) + gauss_noise, 0, 255).astype(np.uint8)

speckle_noise = np.random.normal(0, 0.2, img.shape)
noisy_speckle = np.clip(img.astype(np.float32) + img * speckle_noise, 0, 255).astype(np.uint8)

k = 5

def denoise(noisy_img):
    w = np.clip(wiener(noisy_img.astype(np.float32), (k, k)), 0, 255).astype(np.uint8)
    m = cv2.medianBlur(noisy_img, k)
    g = cv2.GaussianBlur(noisy_img, (k, k), 1.5)
    b = cv2.bilateralFilter(noisy_img, k, 75, 75)
    return w, m, g, b

w_g, m_g, g_g, b_g = denoise(noisy_gauss)
w_s, m_s, g_s, b_s = denoise(noisy_speckle)

plt.figure(figsize=(15, 6))
titles = ["Noisy", "Wiener", "Median", "Gaussian", "Bilateral"]

for i, im in enumerate([noisy_gauss, w_g, m_g, g_g, b_g]):
    plt.subplot(2, 5, i + 1)
    plt.imshow(im, cmap='gray')
    plt.title("Gaussian " + titles[i])
    plt.axis("off")

for i, im in enumerate([noisy_speckle, w_s, m_s, g_s, b_s]):
    plt.subplot(2, 5, i + 6)
    plt.imshow(im, cmap='gray')
    plt.title("Speckle " + titles[i])
    plt.axis("off")

plt.tight_layout()
plt.show()
